# Temas Tratados en el Trabajo Práctico 6

* Modelado de problemas en espacios de estado.

* Algoritmos de planificación hacia adelante y hacia atrás.

* Representación y solución de problemas descritos en lenguaje STRIPS.

* Algoritmo GRAPHPLAN.

* Planificación con restricciones de tiempo y recursos.

* Caminos críticos y tiempos de relajación.

## Ejercicios Teóricos

1. ¿En qué tipo de algoritmos se basa un planificador para encontrar el mejor camino a un estado solución?


2. ¿Qué tres elementos se encuentran dentro de una acción formulada en lenguaje STRIPS? Describa brevemente qué función cumple cada uno.

3. Describa las ventajas y desventajas de desarrollar un algoritmo de planificación hacia adelante y hacia atrás en el espacio de estados.


4. Considere el problema de ponerse uno mismo zapatos y medias. Aplique GRAPHPLAN a este problema y muestre la solución obtenida. Muestre el plan de orden parcial que es solución e indique cuántas linealizaciones diferentes existen para el plan de orden parcial.

5. Se requiere ensamblar una máquina cuyas piezas están identificadas con las letras A, B, C, D y E. El tiempo que se tarda en ensamblar cada pieza es:

* A: 2 semanas

* B: 1 semana

* C: 4 semanas

* D: 3 semanas

* E: 5 semanas

    El orden de ensamblaje de cada pieza requiere que:

* A esté realizado antes que C

* B esté realizado antes que C

* B esté realizado antes que D

* C esté realizado antes que E

* D esté realizado antes que E

    Con esta información:

        5.1 Arme el Plan de Orden Parcial.

        5.2 Encuentre el Camino Crítico.

        5.3 Encuentre los tiempos de relajación.

        5.4 Dibuje un diagrama temporal indicando las tareas y los tiempos de relajación encontrados.

## Ejercicios de Implementación

> Recuerde adjuntar en la presentación el prompt inicial que ha utilizado para cada ejercicio de implementación y si considera que le dio la información completa para resolver el ejercicio o qué cambios adicionales tuvo que pedir de manera iterativa.

6. Suponga que tiene un robot de oficina capaz de moverse y tomar y depositar objetos. El robot solo puede tener un objeto a la vez, pero puede conseguir una *caja* en la que depositar varios objetos. Suponga que programa al robot para *ir a la tienda* a comprarle un *café* y en el camino de vuelta tome una *carta* del *buzón* de la oficina para para que se la traiga junto con el café. Describa en lenguaje STRIPS:

        6.1 El dominio del robot (nombre, predicados y acciones que puede hacer el robot).

        6.2 El problema que se quiere resolver (estado inicial, estado objetivo y objetos del mundo representados).

        6.3 Introduzca el código desarrollado en los puntos anteriores en el [planificador online](http://lcas.lincoln.ac.uk/fast-downward/) y obtenga el plan de acción que tomará el robot para cumplir lo solicitado.

In [ ]:
;; DOMINIO DEL ROBOT:
(define (domain robot-oficina)
  (:requirements :strips :typing)

  (:types
    lugar
    objeto
    caja
  )

  (:predicates
    (robot-en ?l - lugar)
    (conectado ?desde - lugar ?hasta - lugar)
    (objeto-en ?o - objeto ?l - lugar)
    (caja-en ?c - caja ?l - lugar)
    (mano-libre)
    (sosteniendo-objeto ?o - objeto)
    (sosteniendo-caja ?c - caja)
    (objeto-en-caja ?o - objeto ?c - caja)
  )

  ;; Mover el robot (con o sin cosas en la mano)
  (:action mover
    :parameters (?desde - lugar ?hasta - lugar)
    :precondition (and (robot-en ?desde)
                       (conectado ?desde ?hasta))
    :effect (and (robot-en ?hasta)
                 (not (robot-en ?desde)))
  )

  ;; Tomar un objeto suelto del lugar
  (:action tomar-objeto
    :parameters (?o - objeto ?l - lugar)
    :precondition (and (robot-en ?l)
                       (objeto-en ?o ?l)
                       (mano-libre))
    :effect (and (sosteniendo-objeto ?o)
                 (not (objeto-en ?o ?l))
                 (not (mano-libre)))
  )

  ;; Depositar un objeto en el lugar
  (:action soltar-objeto
    :parameters (?o - objeto ?l - lugar)
    :precondition (and (robot-en ?l)
                       (sosteniendo-objeto ?o))
    :effect (and (objeto-en ?o ?l)
                 (not (sosteniendo-objeto ?o))
                 (mano-libre))
  )

  ;; Tomar la caja (ocupa la unica mano)
  (:action tomar-caja
    :parameters (?c - caja ?l - lugar)
    :precondition (and (robot-en ?l)
                       (caja-en ?c ?l)
                       (mano-libre))
    :effect (and (sosteniendo-caja ?c)
                 (not (caja-en ?c ?l))
                 (not (mano-libre)))
  )

  ;; Apoyar la caja en el lugar
  (:action soltar-caja
    :parameters (?c - caja ?l - lugar)
    :precondition (and (robot-en ?l)
                       (sosteniendo-caja ?c))
    :effect (and (caja-en ?c ?l)
                 (not (sosteniendo-caja ?c))
                 (mano-libre))
  )

  ;; Poner el objeto sostenido dentro de la caja (que esta apoyada en el lugar)
  (:action meter-en-caja
    :parameters (?o - objeto ?c - caja ?l - lugar)
    :precondition (and (robot-en ?l)
                       (caja-en ?c ?l)
                       (sosteniendo-objeto ?o))
    :effect (and (objeto-en-caja ?o ?c)
                 (not (sosteniendo-objeto ?o))
                 (mano-libre))
  )

  ;; Sacar un objeto de la caja (apoyada en el lugar)
  (:action sacar-de-caja
    :parameters (?o - objeto ?c - caja ?l - lugar)
    :precondition (and (robot-en ?l)
                       (caja-en ?c ?l)
                       (objeto-en-caja ?o ?c)
                       (mano-libre))
    :effect (and (sosteniendo-objeto ?o)
                 (not (objeto-en-caja ?o ?c))
                 (not (mano-libre)))
  )

  ;; Comprar: entrega el dinero (queda en la tienda) y recibe el cafe en la mano
  (:action comprar
    :parameters (?dinero - objeto ?producto - objeto ?l - lugar)
    :precondition (and (robot-en ?l)
                       (sosteniendo-objeto ?dinero)
                       (objeto-en ?producto ?l))
    :effect (and (objeto-en ?dinero ?l)
                 (not (sosteniendo-objeto ?dinero))
                 (not (objeto-en ?producto ?l))
                 (sosteniendo-objeto ?producto))
  )
)

In [ ]:
;;PROBLEMA A RESOLVER
(define (problem comprar-cafe-y-traer-carta)
  (:domain robot-oficina)

  (:objects
    oficina-personal tienda buzon-oficina - lugar
    dinero cafe carta - objeto
    caja1 - caja
  )

  (:init
    ;; Robot
    (robot-en oficina-personal)
    (mano-libre)

    ;; Mapa (recorrido: oficina -> tienda -> buzon -> oficina)
    (conectado oficina-personal tienda)
    (conectado tienda buzon-oficina)
    (conectado buzon-oficina oficina-personal)

    ;; Objetos en el mundo
    (objeto-en dinero oficina-personal)
    (objeto-en cafe tienda)
    (objeto-en carta buzon-oficina)
    (caja-en caja1 oficina-personal)
  )

  (:goal (and
    (robot-en oficina-personal)
    (objeto-en-caja cafe caja1)
    (objeto-en-caja carta caja1)
    (caja-en caja1 oficina-personal)
    (objeto-en dinero tienda)
  ))
)

## El planificador online devolvió el siguiente plan:
(tomar-objeto dinero oficina-personal)
(mover oficina-personal tienda)
(comprar dinero cafe tienda)
(mover tienda buzon-oficina)
(mover buzon-oficina oficina-personal)
(meter-en-caja cafe caja1 oficina-personal)
(mover oficina-personal tienda)
(mover tienda buzon-oficina)
(tomar-objeto carta buzon-oficina)
(mover buzon-oficina oficina-personal)
(meter-en-caja carta caja1 oficina-personal)
; cost = 11 (unit cost)
 
## Observaciones
No fue el comportamiento esperado, debido a que se esperaria que pase primero por el café, y luego recoja la carta para despues ir a la oficina
Pero, como está planteado, perfiere primero traer el café y luego buscar la carta, sin llevar la caja a ningun lado.

### Por que puede ser lo mas razonable:
-Cargar la caja implica tomarla, moverla y soltarla en cada parada. Si la caja es pesada, voluminosa o incomoda, cada traslado tiene un costo mucho mayor que un desplazamiento simple.
- Si el buzon estan cerca de la oficina, un viaje extra es casi gratis. En una oficina compacta, ir y volver es mas barato que gestionar la caja.

### Que seguiria
- Deberia analizarse los pesos de cada accion, añadiendo un costo No unitario a la acción. 
- Esto dependerá exclusivamente de las distancias que se traten, y de que tanto le complica la situación al robot cargar la caja 

## Prompt Utilizado:
sos un programamdor del lenguaje Strips, que usa la sintaxis PDDL de Fast-downwardy te encargaron hacer el programa que realice la planificación de un robot de oficina.
- El robot es capaz de moverse y tomar y depositar objetos. 
- El robot solo puede tener un objeto a la vez
- El robot puede conseguir una caja en la que depositar varios objetos. Interpretá que la caja está en la oficina, y que el robot no puede agarrar la caja y el objeto a la misma vez, pero si puede colocar el objeto que sostiene en la caja. 
-El robot inicia en "Oficina Personal" (lugar)

El objetivo del robot es el siguiente:
-ir a "la tienda" (Lugar) a comprarle un "café" (objeto) (Esto debe ser interpretado como dejar "dinero" (objeto) y recolectar "Café"(objeto))
-en el camino de vuelta tomar una "carta" (objeto) del "buzón de la oficina" (Lugar)
-Traer junto con el café la carta a "Oficina Personal" (Lugar).

Debes:
-Describir El dominio del robot (nombre, predicados y acciones que puede hacer el robot) en STRIPS
-Describir El problema que se quiere resolver (estado inicial, estado objetivo y objetos del mundo representados)

No debes obtener el plan de acción que tomará el robot, debido a que eso lo realizará un planificador.  SOLO USÁ CARACTERES ASCII, caso contrario falla

### Nota
La ultima aclaración proviene de un problema que tiene la pagina web para tomar caracteres no-ascii. El prompt fue este, y el codigo se generó en un chat limpio despues de diagnosticar este problema


# Bibliografía

[Russell, S. & Norvig, P. (2004) _Inteligencia Artificial: Un Enfoque Moderno_. Pearson Educación S.A. (2a Ed.) Madrid, España](https://www.academia.edu/8241613/Inteligencia_Aritificial_Un_Enfoque_Moderno_2da_Edici%C3%B3n_Stuart_J_Russell_y_Peter_Norvig)

[Poole, D. & Mackworth, A. (2023) _Artificial Intelligence: Foundations of Computational Agents_. Cambridge University Press (3a Ed.) Vancouver, Canada](https://artint.info/3e/html/ArtInt3e.html)